# Analysis Notebook for DeepPlateau
## Evaluating Minimal Discs in $\mathbb{H}^3$

In this notebook, we load the trained Physics-Informed Neural Networks (PINNs) generated in the `train_and_save` notebook. With the optimized models loaded, our primary goal is to analyze the geometry of the resulting minimal surfaces.

In [ ]:
import numpy as np

from src.curves import *
from src.extensions import *
from src.full_models import HyperbolicMinimalSurfacePINN
from src.plotting import plot_error, plot_curve_and_projection, montecarlo_error, plot_H3_surfaces
from src.geometry import minimal_in_Hn_PDE_flat
from src.samplers import MixSampler

from mpl_toolkits.mplot3d import Axes3D
%matplotlib widget

## 1. Loading the Pre-trained Model

Our first step is to load a fully optimized PINN from the `trained_models` directory. 

By calling the `load` method, we automatically reconstruct the neural network's architecture, its geometric extensions, and the boundary defining functions, before injecting the saved, learned weights. 

In the cell below, we load the minimal surface bounded by a star-shaped curve. You can easily switch to analyzing other geometries—such as the gear-shaped curve or the ellipse by uncommenting the respective `model_name` string.

In [ ]:
# load the model
trained_models_path = 'trained_models/'

model_name = trained_models_path + 'CURVE_star_CURVE_PAR_R2.0_petals5.0_r0.3.pt'
# model_name = trained_models_path + 'CURVE_gear_CURVE_PAR_R2.0_amp0.5_sharpness3.0_teeth6.0.pt'
# model_name = trained_models_path + 'CURVE_ellipse_CURVE_PAR_a2.0_b1.0.pt'

model = HyperbolicMinimalSurfacePINN.load(model_name)
untrained_model = HyperbolicMinimalSurfacePINN(**model.kwargs)

## 2. Visualising the boundary curve

To verify that the model and its geometric parameters have been restored correctly, we can extract the boundary curve directly from the loaded model and plot it.

*Note: The 3D plots are interactive. If your notebook environment supports it, you can change the point of view by clicking and dragging the mouse pointer.*

In [ ]:
curve = model.get_curve()
plot_curve_and_projection(curve)

## 3. Visualising the PDE Residual (Before and After)

In the cell below, we plot the PDE error (the squared norm of the tension field) across the unit disc, of an *untrained* version of our PINN. As expected from our mathematical formulation, the error is identically zero exactly at the boundary—even before any learning occurs—thanks to the stereobiharmonic extension. However, the interior error remains high.

In [ ]:
sampler = MixSampler(dtype=untrained_model.kwargs['dtype'])

# plot the error before training on a regular grid
plot_error(
    minimal_in_Hn_PDE_flat(untrained_model),
    dtype=untrained_model.kwargs['dtype'],
    vmin=0,
    vmax=1,
    grid_size=300,
    boundary_linewidth=2.6,
    figsize=(6,5),
    colorbar_label='',
    title=None)

Now, we plot the PDE error for our *trained* model. 

By comparing this heat-map to the baseline above, we can visually verify the neural network's work. The interior residual has been driven down uniformly, indicating a successful deformation into a near-minimal p-immersion.

In [ ]:
# plot the error after training on a regular grid
plot_error(
    minimal_in_Hn_PDE_flat(model),
    dtype=untrained_model.kwargs['dtype'],
    vmin=0,
    vmax=1,
    grid_size=300,
    boundary_linewidth=2.6,
    figsize=(6,5),
    colorbar_label='',
    title=None)

### Monte Carlo Validation

It is important to verify that the loaded model's low PDE residual generalizes across the entire domain. A single grid evaluation provides a good visual sanity check, but a rigorous statistical analysis ensures there are no hidden regions of high error.

In the cell below, we perform a Monte Carlo simulation by drawing 1,000 independent random samples (each containing 16,384 points) from the unit disk. A histogram with a near-zero mean confirms that the surface we restored from the saved weights is near-minimal.

In [ ]:
montecarlo_error(
    minimal_in_Hn_PDE_flat(model),
    num_samples=1000,
    size_samples=2**14,
    figsize=(5,3),
    bins=100,
    title=None,
    xlabel = 'Loss'
)

## 4. Surface Visualisation

Plot of the surfaces produced by the model, before and after training, on the half-space and ball models of $\mathbb{H}^3$. The untrained model is shown in green/yellow, while the trained model is shown in blue/red. 

*Note: These plots are interactive. You can click and drag to change the point of view. Additionally, the `max_theta` parameters in the cell below are currently set to $2\pi - \pi/2$ to create a "cutaway" view, allowing you to see the interior cross-section of the surfaces. Change these to $2\pi$ to view the complete, closed surfaces.*

In [ ]:
plot_H3_surfaces(
    model_A_trained = model,
    grid_size_A=500,
    min_r_A = 0, 
    max_r_A = 1,
    min_theta_A = 0,
    max_theta_A = 2 * np.pi - np.pi / 2, # change this to 2 * np.pi to see the whole surface
    alpha_A = 0.8, # change this to tune the transparency of the trained surface
    
    model_B_untrained = untrained_model,
    grid_size_B=500,
    min_r_B = 0,
    max_r_B = 1,
    min_theta_B = 0,
    max_theta_B = 2 * np.pi - np.pi / 2, # change this to 2 * np.pi to see the whole surface
    alpha_B = 0.8, # change this to tune the transparency of the untrained surface
)